## CSV Relationship Discovery — Anthropic Managed Agent Demo

This example creates a **Skill** and an **Agent** using the Anthropic Managed Agents API to automatically discover primary key / foreign key relationships between CSV datasets.

**How it works:**

1. **Skill** — Provides domain-specific instructions on *how* to perform statistical analysis (uniqueness ratios, inclusion dependencies, value overlap detection)
2. **Agent** — Orchestrates the workflow: loads CSVs, profiles columns, and identifies structural relationships between datasets
3. **Toolset** — The agent uses the built-in code execution environment (`agent_toolset_20260401`) to run Python for data loading, profiling, and statistical computation

In [ ]:
#!pip install anthropic -U

## 1) Create Anthropic client

In [ ]:
from anthropic import Anthropic
from anthropic.lib import files_from_dir
import base64
from pathlib import Path

In [ ]:
client = Anthropic(
    api_key="you_anthropic_api_key"
)

## 2) Create Skill

In [ ]:
skill = client.beta.skills.create(
    display_title="my-data-relation-skill",
    files=files_from_dir("my-data-relation-skill")
)

SKILL_ID = skill.id
print("Skill:", SKILL_ID)

## 3) Create Agent

In [ ]:
system_prompt = """You are a data relationship discovery agent. When given 
one or more CSV datasets, analyze each column's content, data types, 
distributions, and statistics to identify meaningful relationships between 
datasets. Use code to load and profile the data, compute correlations, 
detect matching keys or foreign key patterns"""

agent = client.beta.agents.create(
    name="CSV Data Relationship Agent",
    model={"id": "claude-sonnet-4-6"},
    system=system_prompt,
    tools=[{"type": "agent_toolset_20260401"}],
    skills=[{"type": "custom", "skill_id": SKILL_ID, "version": "latest"}],
)

AGENT_ID = agent.id 
print("Agent:", AGENT_ID)

## 4) Create Enviroment

In [ ]:
environment = client.beta.environments.create(
    name="my-data-relationship-agent-env",
    config={"type": "cloud", "networking": {"type": "unrestricted"}},
)
ENV_ID = environment.id
print("Environment:", ENV_ID)

## 5) Use Agent

### 5.1) Create session and upload csv files to the session

In [ ]:
t1 = client.beta.files.upload(
    file=Path("t1.csv")
)

t2 = client.beta.files.upload(
    file=Path("t2.csv")
)

t3 = client.beta.files.upload(
    file=Path("t3.csv")
)

session = client.beta.sessions.create(
    agent=AGENT_ID,
    environment_id=ENV_ID,
    resources=[
        {
            "type": "file",
            "file_id": t1.id,
            "mount_path": "/workspace/t1.csv",
        },
        {
            "type": "file",
            "file_id":  t2.id,
            "mount_path": "/workspace/t2.csv",
        },
        {
            "type": "file",
            "file_id": t3.id,
            "mount_path": "/workspace/t3.csv",
        },
    ],
)

### 5.2) Call Agent to find relationship between the CSV files

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:

    client.beta.sessions.events.send(
        session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": "find relationship between the datasets t1, t2 and t3."
                    }
                ]
            }
        ]
    )

    response = []

    for event in stream:

        if event.type == "agent.message":
            for block in event.content:
                if getattr(block, "type", None) == "text":
                    response.append(block.text)

        elif event.type == "session.status_idle":
            break

print("\n\n=== FINAL RESPONSE ===\n")
print("".join(response))